# Agent 5: Tracking Expert (트래킹 전문가)

In [ ]:
# Tracking Expert Graph Init
import sys
import os
from pathlib import Path
import json
import time

# 프로젝트 루트 경로 설정
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(project_root))

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"✓ 프로젝트 루트: {project_root}")

# 2. 테스트 이미지 준비
from src.utils import find_data_directory
from src.tools.experts.expert_utils import save_bytes_to_temp_file
from src.tools.experts.expert_utils import call_gemini_text, parse_json_response

# Tracking Expert 관련 프롬프트
from src.prompts.tracking_expert_prompts import get_final_verdict_prompt

try:
    data_dir = find_data_directory()
    # 테스트할 이미지 파일명 (Tracking 분석에 적합한 이미지 권장)
    test_image_name = "Poor_Contact_001.jpg" 
    test_image_path = Path(data_dir) / test_image_name
    
    if not test_image_path.exists():
        image_files = list(Path(data_dir).glob("*.png")) + list(Path(data_dir).glob("*.jpg"))
        if image_files:
            test_image_path = image_files[0]
            print(f"⚠️ 지정된 이미지를 찾을 수 없어 {test_image_path.name}을 사용합니다.")
        else:
            raise FileNotFoundError("테스트할 이미지가 없습니다.")
            
    print(f"✓ 테스트 이미지: {test_image_path}")
    
    with open(test_image_path, 'rb') as f:
        image_data = f.read()
    temp_image_path = save_bytes_to_temp_file(image_data)
    print(f"✓ 임시 이미지 경로: {temp_image_path}")
    
except Exception as e:
    print(f"❌ 오류: {e}")

# 3. 그래프 빌드 및 상태 초기화 (실행 전 필수)
from src.graphs.tracking_expert_graph import build_tracking_expert_graph
from src.nodes.tracking_nodes import TrackingExpertState

print("✓ 그래프 빌드 및 초기 상태 생성 중...")

initial_state = TrackingExpertState(
    messages=[],
    image_path=temp_image_path,
    hotspots=[],
    hotspot_queue=None,
    analysis_results=[],
    
    # Loop 변수 초기화
    current_hotspot=None,
    detector_result=None,
    roi_image_path=None,
    connection_type=None,
    
    # Tracking Specialist Results
    tracking_terminal_result=None,
    tracking_wire_result=None,
    tracking_plug_result=None,
    tracking_pcb_result=None,
    
    # Final Verdict
    verdict_report=None,
    verdict_confidence=None,
    verdict_result=None
)

graph = build_tracking_expert_graph()
print("✓ 완료: initial_state 및 graph 객체 준비됨")

In [ ]:
# Tracking Expert Graph Run
print("=" * 60)
print("Multi-Hotspot Loop 전체 실행 (Tracking Expert)")
print("=" * 60)

start_time = time.time()
try:
    # Recursion Limit을 넉넉히 설정 (Loop 회전 수 고려)
    final_state = graph.invoke(initial_state, config={"recursion_limit": 50})
    elapsed = time.time() - start_time
    
    print(f"\n✓ 전체 실행 완료 (소요시간: {elapsed:.2f}초)")
    
    # 1. 최종 리포트 출력
#     print("\n📊 최종 리포트 (Verdict):")
#     print("-" * 60)
#     print(final_state.get("verdict_report", "결과 없음"))
#     
    # 2. 발견된 Hotspots 확인
#     hotspots = final_state.get("hotspots", [])
#     print(f"\n🔍 발견된 Hotspots: {len(hotspots)}개")
#     for h in hotspots:
#         print(f"   - ID {h.get('id')}: {h.get('damage_type')} (Score: {h.get('severity_score', 0)})")
#     
    # 3. 상세 분석 결과 JSON 출력
#     print("\n📄 [Detailed Analysis Results]")
#     print("-" * 60)
#     analysis_results = final_state.get("analysis_results", [])
    # JSON 직렬화 불가 객체 방지용 default=str 추가
#     print(json.dumps(analysis_results, indent=2, ensure_ascii=False, default=str))
# 
    # 4. 결과 시각화 (Processed Images & Results)
#     print("\n📷 [Processed Images Visualization]")
#     print("-" * 60)
#     
#      from PIL import Image
#      import matplotlib.pyplot as plt
#      import matplotlib.font_manager as fm
#      import os
#      
       # 한글 폰트 설정 (Windows)
#      try:
        # Windows에서 사용 가능한 한글 폰트 찾기
#          font_list = ['Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'Gulim', 'Batang']
#          font_found = None
#          for font_name in font_list:
#              try:
#                  font_path = fm.findfont(fm.FontProperties(family=font_name))
#                  if font_path:
#                      plt.rcParams['font.family'] = font_name
#                      font_found = font_name
#                      break
#              except:
#                  continue
#          
#          if font_found:
#              print(f"✓ 한글 폰트 설정: {font_found}")
#          else:
            # 폰트를 찾지 못한 경우 경고만 출력하고 계속 진행
#              print("⚠️ 한글 폰트를 찾을 수 없습니다. 한글이 제대로 표시되지 않을 수 있습니다.")
#      except Exception as e:
#          print(f"⚠️ 폰트 설정 중 오류: {e}")
#      
#      if not analysis_results:
#          print("No analyzed images found.")
#      else:
#          num_images = len(analysis_results)
#          if num_images > 0:
            # 서브플롯 크기 및 배열 설정
#              cols = min(num_images, 5)
#              rows = (num_images - 1) // cols + 1
#              fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
#              
            # axes가 단일 객체이거나 1차원 배열일 경우 처리
#              if num_images == 1:
#                  axes = [axes]
#              elif rows > 1:
#                  axes = axes.flatten()
#              
            # 빈 서브플롯 숨기기
#              if hasattr(axes, '__len__') and len(axes) > num_images:
#                  for ax in axes[num_images:]:
#                      ax.axis('off')
#              
#              for idx, res in enumerate(analysis_results):
                # Data Extraction adapted for Partial Break structure
#                  h_info = res.get('hotspot_info', {})
#                  hotspot_id = h_info.get('id', '?')
#                  feature = h_info.get('damage_type', 'Unknown')
#                  
#                  roi_path = res.get('roi_image_path') # Fixed: use roi_image_path instead of roi_path
#                  spec_res = res.get('specialist_result', {}) or {}
#                  
#                  is_pb = spec_res.get('is_partial_break', False)
#                  conf = spec_res.get('confidence', 0)
#                  
#                  ax = axes[idx]
#                  
#                  if roi_path and os.path.exists(roi_path):
#                      try:
#                          img = Image.open(roi_path)
#                          ax.imshow(img)
                        # Title 구성: ID, 판정 결과, 신뢰도
#                          title_text = f"ID {hotspot_id}: {feature}\n"
#                          if is_pb:
#                              title_text += f"⚠️ Partial Break ({conf}%)"
#                          else:
#                              title_text += f"Normal / Other ({conf}%)"
#                              
#                          ax.set_title(title_text, fontsize=10, color='red' if is_pb else 'blue')
#                      except Exception as e:
#                          ax.text(0.5, 0.5, "Error Loading", ha='center')
#                          print(f"Error loading image {roi_path}: {e}")
#                  else:
#                      ax.text(0.5, 0.5, "Image Not Found", ha='center')
#                      ax.set_title(f"ID {hotspot_id}")
#                      
#                  ax.axis('off')
#              
#              plt.tight_layout()
#              plt.show()

except Exception as e:
    print(f"❌ 실행 중 오류: {e}")
    import traceback
    traceback.print_exc()